# Week 2 · Lesson 4 — English–French GRU Encoder–Decoder

You will train a **one-layer, unidirectional GRU encoder and a one-layer GRU decoder** on a 10,000-pair slice of OPUS Books. The encoder passes one fixed-length vector to the decoder. The model uses **no attention**.

You will:
1. Prepare 9,000 training pairs and 1,000 separate dev pairs.
2. Train the model and inspect epoch-by-epoch loss and dev BLEU.
3. Read three dev-set translations and compare results across sentence lengths.
4. Write a **120-word reflection** in the final markdown cell.

**All code is complete.** Run every cell in order; there are no code gaps or TODOs. Your task is to interpret the evidence. You may adjust hyperparameters in Section 1 and rerun the notebook from the top.

This small model is a teaching baseline, not a production translator. Low scores or generic translations are possible. Do not assume in advance that longer sentences must score worse or that BLEU must plateau.


## 0 — Install missing dependencies
Run this cell once. Google Colab normally includes PyTorch; only missing packages are installed. The first dataset download requires internet access. OPUS Books is public, so you do not need a Hugging Face token.

A Colab GPU is recommended for the full experiment. CPU execution is supported, but training time depends on your hardware. If an installation asks you to restart the session, do so and rerun from the top.


In [ ]:
import importlib.util
import subprocess
import sys

requirements = {
    "torch": "torch", "numpy": "numpy", "pandas": "pandas",
    "matplotlib": "matplotlib", "datasets": "datasets", "sacrebleu": "sacrebleu",
}
missing = [package for module, package in requirements.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
print("Required packages are available.")


## 1 — Configuration and reproducibility
You use separate English and French vocabularies. Each has reserved IDs for padding, unknown words, beginning of sentence, and end of sentence. Accented characters are retained.

The sample contains pairs of **3–40 tokens in each language**. You filter out longer pairs rather than truncate them, so a dropped clause in a translation cannot be explained by the input having been cut off. This also limits the scope of your conclusions: you are not testing very long documents.

Default training uses full teacher forcing for efficiency. During training the decoder sees the previous correct French token; during BLEU evaluation it sees only its own previous prediction. These are different conditions.


In [ ]:
import os
import re
import html
import random
import time
import unicodedata
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence
from sacrebleu.metrics import BLEU
import sacrebleu

SEED = 42
N_PAIRS, N_DEV = 10000, 1000
MIN_TOKENS, MAX_TOKENS = 3, 40
SRC_VOCAB_LIMIT, TGT_VOCAB_LIMIT = 8000, 8000
MIN_FREQ = 2
EMB, HID = 64, 128
EPOCHS, BATCH_SIZE, LR = 8, 64, 0.003
MAX_DECODE_STEPS = MAX_TOKENS + 1  # Allow up to 40 French tokens followed by EOS.
PAD, UNK, BOS, EOS = 0, 1, 2, 3
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_num_threads(min(2, os.cpu_count() or 1))

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

seed_everything(SEED)
assert 0 < N_DEV < N_PAIRS and EPOCHS >= 1
assert MAX_DECODE_STEPS >= MAX_TOKENS + 1
print("Python:", sys.version.split()[0], "PyTorch:", torch.__version__, "SacreBLEU:", sacrebleu.__version__)
print("Device:", DEVICE, "| Epochs:", EPOCHS, "| EMB:", EMB, "| HID:", HID)
print("Seed is fixed, but exact results can vary across hardware and library versions.")


## 2 — Load and split 10,000 English–French pairs
OPUS Books contains sentence-aligned literary translations. You normalize Unicode and whitespace, lowercase consistently, and tokenize words and punctuation. English and French word order, morphology, and punctuation still differ.

The code removes repeated **normalized English sources** before sampling, so the same English input cannot appear in both train and dev. This is a random sentence split; sentences from the same book can still occur in both sets. It does not measure generalization to unseen books.

A configuration-specific CSV cache avoids repeated downloads. If you work offline, place the valid cache file beside the notebook. No synthetic data is substituted if downloading fails.


In [ ]:
def normalize_text(text):
    text = html.unescape(text)
    text = re.sub(r"<[^>]*>", " ", text)
    return re.sub(r"\s+", " ", unicodedata.normalize("NFC", text).lower()).strip()

def tokenize(text):
    return re.findall(r"\w+|[^\w\s]", text, flags=re.UNICODE)

def detokenize(tokens):
    text = " ".join(tokens)
    text = re.sub(r"\s+([,.;:!?%\)\]»])", r"\1", text)
    text = re.sub(r"([\(\[«])\s+", r"\1", text)
    return text.replace(" ' ", "'").replace(" ’ ", "’").strip()

CACHE = Path(f"opus_books_en_fr_{N_PAIRS}_s{SEED}_len{MIN_TOKENS}-{MAX_TOKENS}_v1.csv")
if CACHE.exists():
    pairs = pd.read_csv(CACHE, keep_default_na=False)
else:
    from datasets import load_dataset
    print("Loading the public OPUS Books English–French data...")
    try:
        books = load_dataset(
            "Helsinki-NLP/opus_books", "en-fr", split="train",
            data_files={"train": "en-fr/train-00000-of-00001.parquet"},
        )
    except Exception as exc:
        raise RuntimeError("Could not load OPUS Books. Check your internet connection or upload the cache CSV, then rerun.") from exc
    rows, seen_english = [], set()
    for item in books:
        translation = item["translation"]
        en, fr = normalize_text(translation["en"]), normalize_text(translation["fr"])
        en_len, fr_len = len(tokenize(en)), len(tokenize(fr))
        if (MIN_TOKENS <= en_len <= MAX_TOKENS and MIN_TOKENS <= fr_len <= MAX_TOKENS
                and en not in seen_english):
            rows.append({"en": en, "fr": fr})
            seen_english.add(en)
    if len(rows) < N_PAIRS:
        raise ValueError(f"Only {len(rows)} eligible pairs; need {N_PAIRS}.")
    pairs = pd.DataFrame(rows).sample(n=N_PAIRS, random_state=SEED).reset_index(drop=True)
    pairs.to_csv(CACHE, index=False)

assert {"en", "fr"}.issubset(pairs.columns) and len(pairs) == N_PAIRS
assert not pairs[["en", "fr"]].isna().any().any()
assert pairs["en"].is_unique
for language in ("en", "fr"):
    assert pairs[language].map(lambda s: isinstance(s, str) and normalize_text(s) == s).all()
    assert pairs[language].map(lambda s: MIN_TOKENS <= len(tokenize(s)) <= MAX_TOKENS).all()

indices = np.random.default_rng(SEED).permutation(len(pairs))
dev_df = pairs.iloc[indices[:N_DEV]].reset_index(drop=True)
train_df = pairs.iloc[indices[N_DEV:]].reset_index(drop=True)
assert set(train_df["en"]).isdisjoint(set(dev_df["en"]))
train_en = [tokenize(s) for s in train_df["en"]]
train_fr = [tokenize(s) for s in train_df["fr"]]
dev_en = [tokenize(s) for s in dev_df["en"]]
dev_fr = [tokenize(s) for s in dev_df["fr"]]
print(f"Selected {len(pairs):,} pairs: {len(train_df):,} train / {len(dev_df):,} dev.")
print("Training example EN:", train_df.iloc[0]["en"])
print("Training example FR:", train_df.iloc[0]["fr"])


## 3 — Build training-only vocabularies and padded batches
Unknown words map to `<unk>` rather than adding entries during evaluation. You will report unknown-token rates because missing vocabulary can damage translation independently of the single-vector bottleneck.

For each target, the decoder input is **`<bos> French words`** and the correct output is **`French words <eos>`**. This one-position shift teaches next-token prediction. Padding is excluded from the loss. Source lengths include their `<eos>` token and are used to skip padding in the encoder.


In [ ]:
SPECIAL = {"<pad>": PAD, "<unk>": UNK, "<bos>": BOS, "<eos>": EOS}
def build_vocab(sentences, limit):
    vocabulary = dict(SPECIAL)
    counts = Counter(token for sentence in sentences for token in sentence)
    for token, count in sorted(counts.items(), key=lambda item: (-item[1], item[0])):
        if count >= MIN_FREQ and token not in vocabulary and len(vocabulary) < limit:
            vocabulary[token] = len(vocabulary)
    return vocabulary

src_vocab = build_vocab(train_en, SRC_VOCAB_LIMIT)
tgt_vocab = build_vocab(train_fr, TGT_VOCAB_LIMIT)
target_words = {idx: token for token, idx in tgt_vocab.items()}

def encode(tokens, vocabulary):
    return [vocabulary.get(token, UNK) for token in tokens]

class TranslationDataset(Dataset):
    def __init__(self, english, french):
        assert len(english) == len(french)
        self.sources = [encode(row, src_vocab) + [EOS] for row in english]
        self.targets = [[BOS] + encode(row, tgt_vocab) + [EOS] for row in french]
    def __len__(self):
        return len(self.sources)
    def __getitem__(self, idx):
        return self.sources[idx], self.targets[idx]

def collate_pairs(batch):
    sources, targets = zip(*batch)
    lengths = torch.tensor([len(s) for s in sources], dtype=torch.long)
    src = pad_sequence([torch.tensor(s, dtype=torch.long) for s in sources], batch_first=True, padding_value=PAD)
    tgt = pad_sequence([torch.tensor(s, dtype=torch.long) for s in targets], batch_first=True, padding_value=PAD)
    return src, lengths, tgt[:, :-1], tgt[:, 1:]

train_ds = TranslationDataset(train_en, train_fr)
dev_ds = TranslationDataset(dev_en, dev_fr)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0,
                          collate_fn=collate_pairs, generator=torch.Generator().manual_seed(SEED))
dev_loader = DataLoader(dev_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, collate_fn=collate_pairs)

def unknown_rate(sentences, vocabulary):
    total = sum(map(len, sentences))
    return sum(token not in vocabulary for row in sentences for token in row) / max(total, 1)

src, lengths, decoder_in, expected = next(iter(dev_loader))
assert src.dtype == decoder_in.dtype == expected.dtype == torch.long
assert torch.equal(lengths, (src != PAD).sum(dim=1))
assert (decoder_in[:, 0] == BOS).all().item()
assert torch.equal(decoder_in[:, 1:], expected[:, :-1])
assert all(train_ds.targets[i][-1] == EOS for i in range(len(train_ds)))
print("Source vocab:", len(src_vocab), "Target vocab:", len(tgt_vocab))
print("Train unknown-token rates: EN", f"{unknown_rate(train_en, src_vocab):.1%}", "FR", f"{unknown_rate(train_fr, tgt_vocab):.1%}")
print("Dev unknown-token rates: EN", f"{unknown_rate(dev_en, src_vocab):.1%}", "FR", f"{unknown_rate(dev_fr, tgt_vocab):.1%}")
print("Source batch:", tuple(src.shape), "Decoder input:", tuple(decoder_in.shape), "Expected output:", tuple(expected.shape))


## 4 — Define a GRU encoder–decoder with no attention
The packed encoder reads the real English tokens and returns one final hidden vector per sentence. The decoder receives that vector as its initial hidden state. It cannot revisit the encoder's sequence of token states: that restriction is the **single-vector bottleneck**.

During training, full teacher forcing lets the decoder process the shifted French input in one call. During translation it starts with `<bos>`, predicts one token at a time, and stops at `<eos>` or the generation limit. It never receives the correct French reference when generating translations.


In [ ]:
class GRUSeq2Seq(nn.Module):
    def __init__(self):
        super().__init__()
        self.src_embedding = nn.Embedding(len(src_vocab), EMB, padding_idx=PAD)
        self.tgt_embedding = nn.Embedding(len(tgt_vocab), EMB, padding_idx=PAD)
        self.encoder = nn.GRU(EMB, HID, num_layers=1, batch_first=True)
        self.decoder = nn.GRU(EMB, HID, num_layers=1, batch_first=True)
        self.output = nn.Linear(HID, len(tgt_vocab))

    def encode(self, src_ids, src_lengths):
        embedded = self.src_embedding(src_ids)
        packed = pack_padded_sequence(embedded, src_lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, context = self.encoder(packed)
        return context  # [1, batch_size, HID]; no encoder token states passed on.

    def forward(self, src_ids, src_lengths, decoder_input):
        context = self.encode(src_ids, src_lengths)
        outputs, _ = self.decoder(self.tgt_embedding(decoder_input), context)
        return self.output(outputs)

    @torch.inference_mode()
    def greedy_decode(self, src_ids, src_lengths, max_steps=MAX_DECODE_STEPS):
        self.eval()
        state = self.encode(src_ids, src_lengths)
        current = torch.full((src_ids.size(0), 1), BOS, dtype=torch.long, device=src_ids.device)
        finished = torch.zeros(src_ids.size(0), dtype=torch.bool, device=src_ids.device)
        generated = []
        for _ in range(max_steps):
            output, state = self.decoder(self.tgt_embedding(current), state)
            logits = self.output(output[:, -1])
            logits[:, PAD] = -torch.inf
            logits[:, BOS] = -torch.inf
            token = logits.argmax(dim=-1)
            token = torch.where(finished, torch.full_like(token, EOS), token)
            generated.append(token)
            finished |= token.eq(EOS)
            current = token.unsqueeze(1)
            if finished.all().item():
                break
        return torch.stack(generated, dim=1).cpu().tolist(), finished.cpu().tolist()

seed_everything(SEED)
model = GRUSeq2Seq().to(DEVICE)
src, lengths, decoder_in, expected = next(iter(dev_loader))
with torch.no_grad():
    logits = model(src.to(DEVICE), lengths, decoder_in.to(DEVICE))
    context = model.encode(src.to(DEVICE), lengths)
    extra_src = torch.nn.functional.pad(src, (0, 3), value=PAD)
    extra_context = model.encode(extra_src.to(DEVICE), lengths)
assert logits.shape == (*expected.shape, len(tgt_vocab))
assert context.shape == (1, len(src), HID)
assert torch.allclose(context, extra_context, atol=1e-5)
print("Context shape:", tuple(context.shape), "| Logits:", tuple(logits.shape))
print("Trainable parameters:", f"{sum(p.numel() for p in model.parameters()):,}")
print("Target shifting, model shapes, and encoder padding checks passed.")


## 5 — Define loss, greedy evaluation, and corpus BLEU
You average cross-entropy over **non-padding target tokens**, including `<eos>`. The dev loss is teacher-forced; BLEU is computed from freely generated translations. A falling dev loss does not guarantee better free-running translation.

BLEU measures n-gram overlap with the supplied reference, with a penalty for overly short output. It is reported on a **0–100 scale**, not as percent accuracy. You use SacreBLEU corpus BLEU, tokenizer `13a`, on consistently lowercased, detokenized predictions and references. The score is specific to this filtered dev set and preprocessing; it is not directly comparable with published benchmarks.

`evaluate_dev` calculates one corpus score for all dev hypotheses; it does not average sentence-level BLEU. The printed signature records the metric settings.


In [ ]:
loss_fn = nn.CrossEntropyLoss(ignore_index=PAD, reduction="sum")
bleu_metric = BLEU(tokenize="13a", lowercase=True, smooth_method="exp")
dev_references = [detokenize(tokens) for tokens in dev_fr]

def decode_words(ids):
    words = []
    for idx in ids:
        if idx == EOS:
            break
        if idx not in (PAD, BOS):
            words.append(target_words.get(idx, "<unk>"))
    return detokenize(words)

@torch.inference_mode()
def evaluate_dev():
    model.eval()
    total_loss, total_tokens = 0.0, 0
    hypotheses, ended = [], []
    for src, lengths, decoder_in, expected in dev_loader:
        src, decoder_in, expected = src.to(DEVICE), decoder_in.to(DEVICE), expected.to(DEVICE)
        logits = model(src, lengths, decoder_in)
        total_loss += loss_fn(logits.reshape(-1, len(tgt_vocab)), expected.reshape(-1)).item()
        total_tokens += expected.ne(PAD).sum().item()
        generated, stopped = model.greedy_decode(src, lengths)
        hypotheses.extend(decode_words(row) for row in generated)
        ended.extend(stopped)
    assert len(hypotheses) == len(dev_references) == N_DEV
    score = bleu_metric.corpus_score(hypotheses, [dev_references])
    return total_loss / total_tokens, score, hypotheses, np.array(ended, dtype=bool)

def synchronize():
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()

# Check the shifting/loss contract with a tiny, non-padding target.
assert EOS != PAD and BOS != EOS and UNK != PAD
assert all(len(row) > 0 for row in dev_references)
print("Evaluation ready: greedy predictions only; dev references are used for scoring, not decoding.")


## 6 — Train and record dev BLEU after every epoch
Training loss excludes padding; gradient clipping reduces the risk of exploding gradients. Each epoch prints train loss, dev loss, dev BLEU, and elapsed time for training plus dev evaluation.

You retain the checkpoint with the highest dev BLEU (lower dev loss breaks ties). The later examples and diagnostics use this checkpoint. Because you use dev scores for selection, they are not independent test results.


In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
history = []
best_key, best_state, best_epoch = None, None, None
for epoch in range(1, EPOCHS + 1):
    model.train()
    synchronize()
    start = time.perf_counter()
    train_loss_sum, train_token_count = 0.0, 0
    for src, lengths, decoder_in, expected in train_loader:
        src, decoder_in, expected = src.to(DEVICE), decoder_in.to(DEVICE), expected.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        logits = model(src, lengths, decoder_in)
        batch_loss = loss_fn(logits.reshape(-1, len(tgt_vocab)), expected.reshape(-1))
        token_count = expected.ne(PAD).sum()
        loss = batch_loss / token_count
        if not torch.isfinite(loss):
            raise RuntimeError(f"Non-finite training loss in epoch {epoch}.")
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        train_loss_sum += batch_loss.item()
        train_token_count += token_count.item()
    dev_loss, dev_score, _, ended = evaluate_dev()
    synchronize()
    elapsed = time.perf_counter() - start
    row = {"epoch": epoch, "train_loss": train_loss_sum / train_token_count,
           "dev_loss": dev_loss, "dev_bleu": dev_score.score, "seconds": elapsed,
           "no_eos_rate": 1 - ended.mean()}
    history.append(row)
    key = (dev_score.score, -dev_loss)
    if best_key is None or key > best_key:
        best_key, best_epoch = key, epoch
        best_state = {name: tensor.detach().cpu().clone() for name, tensor in model.state_dict().items()}
    print(f"Epoch {epoch}/{EPOCHS} | train loss={row['train_loss']:.4f} | dev loss={dev_loss:.4f}"
          f" | dev BLEU={dev_score.score:.2f} | no EOS={row['no_eos_rate']:.1%} | time={elapsed:.1f}s", flush=True)

model.load_state_dict(best_state)
final_dev_loss, final_bleu, dev_hypotheses, dev_ended = evaluate_dev()
print(f"\nRestored best dev checkpoint: epoch {best_epoch}")
print("Dev corpus BLEU:", f"{final_bleu.score:.2f}", "(0–100 scale)")
print("Metric signature:", bleu_metric.get_signature())
print("BLEU details:", final_bleu)
assert np.isclose(final_bleu.score, best_key[0])
assert len(history) == EPOCHS and np.isfinite(final_bleu.score)


## 7 — Inspect the learning curves
Compare teacher-forced loss with free-running dev BLEU. Does BLEU improve, flatten, or fluctuate? Report the pattern you actually see. An eight-epoch run may not show a plateau.

Lower cross-entropy means the model assigns more probability to reference tokens under teacher forcing. It does not mean that every generated sentence is complete or fluent.


In [ ]:
history_df = pd.DataFrame(history)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(history_df["epoch"], history_df["train_loss"], "o-", label="Train")
axes[0].plot(history_df["epoch"], history_df["dev_loss"], "o-", label="Dev")
axes[0].set(title="Teacher-forced loss", xlabel="Epoch", ylabel="Cross-entropy per target token")
axes[0].legend()
axes[1].plot(history_df["epoch"], history_df["dev_bleu"], "o-", color="#15803d")
axes[1].set(title="Free-running dev BLEU", xlabel="Epoch", ylabel="Corpus BLEU (0–100)")
for ax in axes:
    ax.set_xticks(history_df["epoch"])
    ax.grid(alpha=0.2)
fig.tight_layout()
plt.show()


## 8 — Inspect three sample translations
You inspect one short (3–10 tokens), one medium (11–20), and one longer (21–40) English input when each group is present. Examples are selected by a fixed index rule, **not by translation quality**. The printed French reference is the human translation, not an expected exact match.

Look for generic phrases, repetition, missing content, incorrect word choices, and early endings. An unfamiliar French phrase can have a valid alternative translation, so explain specific evidence rather than equating every wording difference with an error. A generation-limit message means decoding stopped without `<eos>`; do not mistake this for an architectural explanation by itself.


In [ ]:
source_lengths = np.array([len(row) for row in dev_en])
length_groups = [("Short", 3, 10), ("Medium", 11, 20), ("Longer", 21, MAX_TOKENS)]
sample_indices = []
for name, low, high in length_groups:
    candidates = np.flatnonzero((source_lengths >= low) & (source_lengths <= high))
    if len(candidates):
        sample_indices.append(int(candidates[0]))
for idx in range(len(dev_df)):
    if len(sample_indices) >= 3:
        break
    if idx not in sample_indices:
        sample_indices.append(idx)
assert len(sample_indices) == 3

for number, idx in enumerate(sample_indices, 1):
    print(f"\nSample {number} | dev row {idx} | source length={source_lengths[idx]} tokens")
    print("English:  ", dev_df.iloc[idx]["en"])
    print("Reference:", dev_references[idx])
    print("Model:    ", dev_hypotheses[idx] or "[empty translation]")
    if not dev_ended[idx]:
        print("Note: generation limit reached without EOS.")


## 9 — Compare evidence across sentence lengths
The table reports corpus BLEU separately within each length group, plus the number of examples, English unknown-token rate, generation-limit rate, and output/reference token-length ratio. Ratios well below 1 can flag short output; low n-gram overlap can reflect missing or wrong phrases. Neither measure diagnoses the cause on its own.

**Interpretation limits:** sentence difficulty, vocabulary coverage, limited training, literary style, greedy decoding, and the single-vector representation can all affect performance. Different length groups contain different content and reference lengths, so their BLEU scores are descriptive rather than a controlled test of the bottleneck. A stronger next experiment would add attention while holding the data, tokenization, dimensions, training budget, and evaluation procedure constant.


In [ ]:
diagnostic_rows = []
for name, low, high in length_groups:
    selected = np.flatnonzero((source_lengths >= low) & (source_lengths <= high))
    if not len(selected):
        continue
    hypotheses = [dev_hypotheses[i] for i in selected]
    references = [dev_references[i] for i in selected]
    score = bleu_metric.corpus_score(hypotheses, [references]).score
    output_tokens = sum(len(tokenize(text)) for text in hypotheses)
    reference_tokens = sum(len(dev_fr[i]) for i in selected)
    source_tokens = [dev_en[i] for i in selected]
    diagnostic_rows.append({"group": f"{name} ({low}–{high})", "pairs": len(selected),
        "BLEU": score, "output/ref_token_ratio": output_tokens / reference_tokens,
        "EN_unknown_rate": unknown_rate(source_tokens, src_vocab),
        "no_EOS_rate": 1 - dev_ended[selected].mean()})
diagnostics = pd.DataFrame(diagnostic_rows)
print(f"Best checkpoint: epoch {best_epoch}; full dev BLEU={final_bleu.score:.2f}")
print(diagnostics.round(4).to_string(index=False))
print("\nUse your examples, BLEU trend, and diagnostics as evidence. Do not invent dropped clauses or a BLEU plateau.")


## Submission checklist and references
Restart your runtime/kernel and run all cells in order. Submit this `.ipynb` with:
- Every code cell executed successfully, including epoch-by-epoch training logs.
- A dev corpus BLEU score, the plots, and three sample translations.
- The length-group diagnostic table.
- Your own 120-word reflection in the final markdown cell.

You do not need to implement attention in this lab. In the next lesson, you can test whether allowing the decoder to revisit encoder token states improves translation.

**References**
- Tiedemann, J. (2012). *Parallel Data, Tools and Interfaces in OPUS.* [ACL Anthology](https://aclanthology.org/L12-1246/). [OPUS Books dataset](https://huggingface.co/datasets/Helsinki-NLP/opus_books).
- Cho et al. (2014). *Learning Phrase Representations using RNN Encoder–Decoder for Statistical Machine Translation.* [Paper](https://aclanthology.org/D14-1179/).
- Bahdanau, Cho, and Bengio (2015). *Neural Machine Translation by Jointly Learning to Align and Translate.* [Paper](https://arxiv.org/abs/1409.0473).
- Post, M. (2018). *A Call for Clarity in Reporting BLEU Scores.* [Paper](https://aclanthology.org/W18-6319/). [SacreBLEU implementation](https://github.com/mjpost/sacrebleu).
- [PyTorch seq2seq tutorial](https://docs.pytorch.org/tutorials/intermediate/seq2seq_translation_tutorial.html). This notebook uses a plain decoder with no attention.


## Reflection — 120 words

Write approximately 120 words addressing **all four prompts**:

1. **Concrete evidence:** Identify a specific translation error or learning-curve pattern you observed. Cite a sample/dev row or epoch. Explain how it is consistent with a possible single-vector bottleneck; acknowledge another plausible cause.
2. **Impact on BLEU:** Cite your dev BLEU value or trend and explain how the observed missing, wrong, or overly short content could affect n-gram overlap or the brevity penalty. If BLEU did not plateau, say so.
3. **Why attention could help:** Explain how access to encoder token states could reduce reliance on one fixed-length context vector.
4. **Next experiment:** Propose one controlled experiment and name the outcome you would compare.

**Your reflection:**

*Replace this line with your own 120-word response. No sample answer is provided.*
